In [2]:
# GLOBAL #

import random
import pandas as pd
from collections import defaultdict, Counter
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split

random.seed(123)
np.random.seed(123)


In [3]:
# HELPER FUNCTIONS #

# collapse sequence by self-transitions (e.g. [aaaabbbbcccdddccc] = [abcdc]) for collapsed context model
def collapseLength(seq):

    collapsed = []

    for s in seq:
        if not collapsed or collapsed[-1] != s:
            collapsed.append(s)

    return collapsed

# check if tuple has consecutive repeated states (we want to ignore these combinations in collapsed context model transition matrix)
def hasConsecutiveRepeats(states):

    for i in range(len(states) - 1):
        if states[i] == states[i + 1]:
            return True
        
    return False

# find indices of first state in each state run within sequence (e.g., [aaaabbbbcccdddccccc] = [0, 4, 8, 11, 14])
def uniqueIdxs(seq):

    lasts = []
    runChar = seq[0]

    for i, s in enumerate(seq[1:], start=1):
        if s != runChar:
            lasts.append(i - 1)
            runChar = s

    lasts.append(len(seq) - 1)
    return lasts

# expand  numeric vector into repetitions corresponding to each number value (e.g., [0, 4, 8, 11, 14] = [0, 0, 0, 0, 4, 4, 4, 4, 8, 8, 8, 11, 11, 11, 14, 14, 14]) 
# for matching strings to indices in collapsed context model 
def expandVec(vec):

    if not vec:
        return []
    
    result = []

    vecIDX = 0

    for pos in range(vec[-1] + 1):
        while vecIDX < len(vec) - 1 and pos > vec[vecIDX]:
            vecIDX += 1
        result.append(vec[vecIDX])

    return result

#calculate weighted log likelihood of test transitions (unnormalized) under train model (normalized)
def averageLL(trainProbs, testProbs):

    ll = 0.0 # start at zero

    weight = 0.0

    for s, nexts in testProbs.items(): #for each item in the test transition 
        trainContext = trainProbs.get(s) #find matching train model probability

        if trainContext is None:
            continue  
                             # if context unseen in train, skip
        for s2, p_test in nexts.items():
            p_train = trainContext.get(s2)

            if p_train is None or p_train <= 0:
                continue  # if transition unseen in train, skip
                             
            ll += p_test * np.log(p_train)
            weight += p_test

    if weight == 0:
        return np.nan
    
    return ll / weight

## main sampling functions ##

""" general pipeline:

normalize the training set because we are essentially building a smoothed model that we can use to evaluate test set fit
keep test set as count data because we are comparing how well it fits to our smoothed model

 """

##### Lagged Context Markov Sampling--bad performance in synthetic eval, not used #####

# def laggedTransNorm(states, lines, k, smoother, lag):

#     transitions = defaultdict(Counter)
    
#     #for k=0 version, transition matrix turns into simple marginal state distributions
#     if k == 0:
#         for state in states:
#             transitions[()][state] = smoother #add laplace smoother to each curr state

#         for seq in lines:
#             for curr in seq:
#                 transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

#         total = sum(transitions[()].values())

#         return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

#     for prev in product(states, repeat=k): #add laplace smoother to every combination of prev & curr states

#         for curr in states:
#             transitions[prev][curr] = smoother

#     for seq in lines:
#         seqLength = len(seq)

#         for t in range(k * lag, seqLength):
#             prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k))) # important line: looks back at k previous states each spaced one lag apart, starting from t-1
#             curr = seq[t] # curr state is at time t
#             transitions[prev][curr] += 1 #add 1 count to this prev & curr combination

#     transNorm = {}

#     for cond, counter in transitions.items():
#         total = sum(counter.values())
#         transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize counts by total number of transitions

#     return transNorm

# def laggedTransCount(lines, k, lag):

#     transitions = defaultdict(Counter)

#     # if k is zero, just count instances of each state
#     if k == 0:
#         for seq in lines:
#             for curr in seq:
#                 transitions[()][curr] += 1
#         return transitions
    

#     for seq in lines:
#         seqLength = len(seq)
#         for t in range(k * lag, seqLength):
#             prev = tuple(seq[t - lag * i - 1] for i in reversed(range(k))) # important line: looks back at k previous states each spaced one lag apart, starting from t-1
#             curr = seq[t] # curr state is at time t
#             transitions[prev][curr] += 1  #add 1 count to this prev & curr combination

#     return transitions

### Collapsed Context Markov Sampling ###

def collapsedTransNorm(states, lines, k, smoother):
    
    transitions = defaultdict(Counter)

    #for k=0 version, transition matrix turns into simple marginal state distributions
    if k == 0:
        for state in states:
            transitions[()][state] = smoother #add laplace smoother to each curr state

        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1 #add one count in corresponding transition cell for each instance of each state

        total = sum(transitions[()].values())

        return {(): {s: c / total for s, c in transitions[()].items()}} #normalize

    for prev in product(states, repeat=k):

        if hasConsecutiveRepeats(prev): #if prev tuple has consecutive repeats, skip (because this cannot be observed when self-transitions are collapsed)
            continue

        for curr in states:
            transitions[prev][curr] = smoother #add laplace smoother to every valid combination of prev & curr states

    for seq in lines:

        idxs = uniqueIdxs(seq) #find indices of unique states in sequence
        idxsExpanded = expandVec(idxs) #expand indices for mapping

        for t in range(1, len(seq)):

            idxsCollapsed = idxs.index(idxsExpanded[t - 1]) #find corresponding index of each token

            if idxsCollapsed < k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    transNorm = {}

    for cond, counter in transitions.items():
        total = sum(counter.values())
        transNorm[cond] = {s: c / total for s, c in counter.items()} #normalize 

    return transNorm

def collapsedTransCount(lines, k):

    transitions = defaultdict(Counter)

    # if k is zero, just count instances of each state
    if k == 0:
        for seq in lines:
            for curr in seq:
                transitions[()][curr] += 1

        return transitions
    
    for seq in lines:
        idxs = uniqueIdxs(seq)
        idxsExpanded = expandVec(idxs)

        for t in range(1, len(seq)):
            idxsCollapsed= idxs.index(idxsExpanded[t - 1])

            if idxsCollapsed< k - 1:
                continue

            prevIdxsUnique = idxs[idxsCollapsed- (k - 1):idxsCollapsed+ 1]
            prev = tuple(seq[i] for i in prevIdxsUnique)
            curr = seq[t]
            transitions[prev][curr] += 1

    return transitions

### sample over k's ###

def fitModel(species, i, trainLines, testLines, kMax, smootherSet):
 
    results = []

    #find unique states in corpus
    states = sorted({char for line in trainLines for char in line})

    for k in range(0, kMax + 1):

        for smoother in smootherSet:

            train_trans = collapsedTransNorm(states = states, lines=trainLines, k=k, smoother=smoother)

            test_trans  = collapsedTransCount(lines=testLines, k=k)

            ll = averageLL(train_trans, test_trans)

            results.append({
                'species': species, 'k': k,
                'sim': i,
                'smoother': smoother,
                'log_likelihood': ll,
                'num_train_states': len(train_trans),
                'num_test_states':  len(test_trans),
            })
            print(f"{species} sim {i}: k={k}")

    return pd.DataFrame(results)


In [4]:
# DO IT! #

birds = ("bird1", "bird2", "bird3", "bird4", "bird5", "bird6")

# params to test
nIter = 100
kmax= 5
smoothers=[0.001, 0.01, 0.1]
cropped = True

#run iterations
results = []

for bird in birds:

    for i in range(1, nIter+1):

        #random train/test split in each iteration

        if cropped == True:
            filename = f"../bird-data/raw/{bird}_cropped_full.txt"  
                     
        else: 
            filename = f"../bird-data/raw/{bird}_full.txt"

        with open(filename) as f:
            lines = f.read().splitlines() 

            lines = [x for x in lines if x] # remove blank lines (sometimes cropping window is the whole string)
 
        train_lines, test_lines = train_test_split(lines, test_size=0.2)
        
        # run markov sampling on this split and save results
        iModel = fitModel(species = bird, i = i, trainLines=train_lines, testLines=test_lines, kMax = kmax, smootherSet = smoothers)

        results.append(iModel)


#save as .csv
results_df = pd.concat(results, ignore_index=True)

if cropped == True:
    savename = "../bird-data/birdResultsCropped.csv" 
else:
    savename = "../bird-data/birdResultsFull.csv"

results_df.to_csv(savename, index=False)
        

bird1 sim 1: k=0
bird1 sim 1: k=0
bird1 sim 1: k=0
bird1 sim 1: k=1
bird1 sim 1: k=1
bird1 sim 1: k=1
bird1 sim 1: k=2
bird1 sim 1: k=2
bird1 sim 1: k=2
bird1 sim 1: k=3
bird1 sim 1: k=3
bird1 sim 1: k=3
bird1 sim 1: k=4
bird1 sim 1: k=4
bird1 sim 1: k=4
bird1 sim 1: k=5
bird1 sim 1: k=5
bird1 sim 1: k=5
bird1 sim 2: k=0
bird1 sim 2: k=0
bird1 sim 2: k=0
bird1 sim 2: k=1
bird1 sim 2: k=1
bird1 sim 2: k=1
bird1 sim 2: k=2
bird1 sim 2: k=2
bird1 sim 2: k=2
bird1 sim 2: k=3
bird1 sim 2: k=3
bird1 sim 2: k=3
bird1 sim 2: k=4
bird1 sim 2: k=4
bird1 sim 2: k=4
bird1 sim 2: k=5
bird1 sim 2: k=5
bird1 sim 2: k=5
bird1 sim 3: k=0
bird1 sim 3: k=0
bird1 sim 3: k=0
bird1 sim 3: k=1
bird1 sim 3: k=1
bird1 sim 3: k=1
bird1 sim 3: k=2
bird1 sim 3: k=2
bird1 sim 3: k=2
bird1 sim 3: k=3
bird1 sim 3: k=3
bird1 sim 3: k=3
bird1 sim 3: k=4
bird1 sim 3: k=4
bird1 sim 3: k=4
bird1 sim 3: k=5
bird1 sim 3: k=5
bird1 sim 3: k=5
bird1 sim 4: k=0
bird1 sim 4: k=0
bird1 sim 4: k=0
bird1 sim 4: k=1
bird1 sim 4: k